In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

experiments = [
    {'name': 'TwoBandit', 'experiment': 'exp1', 'num_options': 2, 'held-out': False},
    {'name': 'TwoBandit', 'experiment': 'exp2', 'num_options': 2, 'held-out': False},
    {'name': 'DriftingBandit', 'experiment': 'exp0', 'num_options': 4, 'held-out': False},
    {'name': 'HorizonSomer', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonWaltz', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonSade', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'HorizonFeng', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'ChangingBandit', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'MaggiesFarm', 'experiment': 'exp0', 'num_options': 3, 'held-out': True},
]

experiment_labels = {
    "TwoBandit_exp1": "TwoBandit - Exp.1",
    "TwoBandit_exp2": "TwoBandit - Exp.2",
    "DriftingBandit": "Drifting Bandit",
    "HorizonSomer": "Horizon Somer",
    "HorizonWaltz": "Horizon Waltz",
    "HorizonSade": "Horizon Sade",
    "HorizonFeng": "Horizon Feng",
    "ChangingBandit": "Changing Bandit",
    "MaggiesFarm": "Maggie's Farm",
    "Mean": "Mean",
}

data_paths = ['LLMPredict/Base/', 'LLMPredict/FineTuned/', 'RescorlaWagnerResults/', 'EvolvedCogModel/FineTuned/', 'EvolvedCogModel/Base/']

repo_url = (
    "https://huggingface.co/datasets/"
    "cObliUvA/llm-guided-modelling-results/resolve/main"
)

In [ ]:
all_rows = []

for dp in data_paths[2:]:

    df = pd.read_csv(f"{repo_url}/{dp}Results_summary.csv")

    df["experiment_id"] = np.where(
        df["name"].eq("TwoBandit"),
        df["name"] + "_" + df["experiment"].astype(str),
        df["name"]
    )

    if "RescorlaWagner" in dp:
        model = "Rescorla-Wagner"
    elif "FineTuned" in dp:
        model = "FineTuned"
    else:
        model = "Base"

    df["model"] = model
    df["gap"] = df["test_nll"] - df["train_nll"]

    all_rows.append(df)

summary = pd.concat(all_rows, ignore_index=True)


# ----------- EXPERIMENT ORDER FROM METADATA -----------
experiment_ids = []

for exp in experiments:

    if exp["name"] == "TwoBandit":
        exp_id = f"{exp['name']}_{exp['experiment']}"
    else:
        exp_id = exp["name"]

    experiment_ids.append(exp_id)


# ----------- ADD EQUAL-WEIGHT MEAN ACROSS EXPERIMENTS -----------
mean_rows = []

for model in summary["model"].unique():

    subset = summary[summary["model"] == model]

    mean_rows.append({
        "experiment_id": "Mean",
        "model": model,
        "gap": subset["gap"].mean()
    })

summary = pd.concat(
    [summary, pd.DataFrame(mean_rows)],
    ignore_index=True
)

experiments_order = experiment_ids + ["Mean"]

# ----------- INCLUDED / HELD-OUT BOUNDARY -----------
first_held_out_idx = next(
    i for i, exp in enumerate(experiments)
    if exp["held-out"]
)

held_out_separator_x = first_held_out_idx - 0.5

included_center = (first_held_out_idx - 1) / 2

held_out_center = (
    first_held_out_idx + len(experiments) - 1
) / 2


# ----------- PIVOT -----------
gap = summary.pivot(
    index="experiment_id",
    columns="model",
    values="gap"
).reindex(experiments_order)


# ----------- STYLE -----------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
})


# ----------- PLOT -----------
fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(experiments_order))

offsets = {
    "Rescorla-Wagner": -0.25,
    "Base": 0.0,
    "FineTuned": 0.25
}

colors = {
    "Rescorla-Wagner": "#4C78A8",
    "Base": "#F58518",
    "FineTuned": "#54A24B"
}


# -----------  LOLLIPOPS FROM ZERO -----------
for model in offsets:

    xpos = x + offsets[model]
    values = gap[model].values

    # sticks
    ax.vlines(
        xpos,
        0,
        values,
        color=colors[model],
        linewidth=4,
        alpha=0.5
    )

    # endpoints
    ax.scatter(
        xpos,
        values,
        s=100,
        color=colors[model],
        zorder=3
    )


# ----------- REFERENCE LINE -----------
ax.axhline(
    0,
    color="black",
    linewidth=1.5
)


# ----------- INCLUDED / HELD-OUT SEPARATOR -----------
ax.axvline(
    held_out_separator_x,
    color="#B0B0B0",
    linestyle="-",
    linewidth=1.0,
    alpha=0.8
)


# ----------- SEPARATOR BEFORE MEAN -----------
mean_separator_x = len(experiments_order) - 1.5

ax.axvline(
    mean_separator_x,
    color="#909090",
    linestyle="-",
    linewidth=2.0,
    alpha=0.8
)

# ----------- INCLUDED / HELD-OUT LABELS -----------
ax.text(
    included_center,
    1.01,
    "Included",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=16
)

ax.text(
    held_out_center,
    1.01,
    "Held-out",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=16
)


# ----------- AXIS FORMATTING -----------
ax.set_xticks(x)

ax.set_xticklabels(
    [experiment_labels[exp] for exp in experiments_order],
    rotation=30,
    ha="right"
)

ax.set_ylabel("Generalization Gap")


# ----------- LEGEND -----------
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color=colors["Rescorla-Wagner"],
        markersize=10,
        linewidth=3,
        label="Rescorla-Wagner"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color=colors["Base"],
        markersize=10,
        linewidth=3,
        label="OpenEvolve Model - Base"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color=colors["FineTuned"],
        markersize=10,
        linewidth=3,
        label="OpenEvolve Model - Fine-Tuned"
    ),
]

ax.legend(handles=legend_handles)

plt.tight_layout()
plt.savefig(
    "../Images/GeneralizationGap.png",
    transparent=True,
    bbox_inches="tight",
    dpi=300
)
plt.show()